# Deposit Attrition EDA — v4

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

Two jobs: close the v3 gaps, and build the **twelve-signal evidence pack** as a
self-contained HTML report.

## The constraint that shapes this notebook

Data before 2024 is not reliable and none can be added, so the panel is 31 months —
permanently. A 12-month self-baseline therefore always eats 2024 and always discards
about a quarter of the attriters (12,138 of 16,384), and the new-entity burn-in never
goes away.

**The answer is not more history — it is peer-relative normalisation.** Compare each
customer to its size-decile peers in the *same calendar month* rather than to its own
past. No pre-window needed, every attriter usable, and a month where everyone's counts
are inflated inflates the peer median too, which cancels the burn-in for free. Both
normalisations are computed; agreement is the sensitivity check, disagreement is a finding.

## What else changes

| | |
|---|---|
| **v3 measured precision on eight features and left out the two that separate earliest** | §4 scores all twelve. `cpty_new_out` and `fin_new_out` get a **zero rule** — "went to zero this month" is the natural operating rule for a count, and a ratio threshold cannot express it |
| Lift was never reported | Precision looks bad at a 0.9% base rate. §4 reports **lift**, which is the honest read on whether a signal carries information |
| Rails were only ever shares | §5a ranks the rails by *when each one goes*, on amounts |
| "Fewer or smaller" was untestable | `avg_ticket_out` is derived, and §5c decomposes the fall into count versus ticket |
| Net flow was only a level | §5d reports the **share of customers whose net flow is actually negative** |
| No benchmark | §5e measures the incumbent 30% rule: how much warning it gives, and how often it never fires at all |
| "No single rule works" was where v3 stopped | §6 tests whether the signals **stack**, as an unweighted count with no fitting |

## Output

§7 writes **`PKG_Attrition_Signals.html`** — one section per signal, each with cards, an
event curve, a lift-by-month chart, and the deep-dive table where one exists. Inline SVG,
no CDN, no JavaScript: it opens offline on a bank laptop.

**Runs in minutes.** Reads v2's panels and v3's labels; rebuilds nothing.

## 0 · Configuration and the twelve-signal spec

In [ ]:
# =====================================================================
# 0 · CONFIGURATION — v4
# =====================================================================
# Two jobs: close the v3 gaps, and produce the twelve-signal evidence pack.
#
# THE CONSTRAINT THAT SHAPES THIS NOTEBOOK. Data before 2024 is not
# reliable, so the panel can never be extended backwards. A 12-month
# self-baseline therefore always consumes 2024 and always costs ~4,200 of
# the 16,384 attriters, and the new-entity burn-in is permanent.
#
# The answer is not more history. It is PEER-RELATIVE normalisation:
# compare a customer to its size-and-segment peers in the SAME calendar
# month rather than to its own past. No pre-window needed, every attriter
# usable, and a month where everyone's counts are inflated inflates the
# peer median too - which neutralises the burn-in for free.
#
# Both normalisations are computed. Agreement between them is the
# sensitivity check; disagreement is a finding.
from pathlib import Path

HDFS_V2  = "hdfs://nameservice1/user/pk36814/attrition_v2"
HDFS_V3  = "hdfs://nameservice1/user/pk36814/attrition_v3"
HDFS_DIR = "hdfs://nameservice1/user/pk36814/attrition_v4"
OUT_DIR  = Path("/projects/DSI/sa15474/repos/pkg/eda/attrition_v4")

DATE_START = "2024-01-01"          # hard floor: nothing before this is trusted
DATE_END   = "2026-07-31"

MAX_ROWS   = 60
ZERO_TOL   = 1.0

# ── Event study ───────────────────────────────────────────────────────
STUDY_DEFS  = ["A_full_exit", "B_bal_exit"]
EVENT_PRE   = 12
EVENT_POST  = 3
BASE_WINDOW = (-12, -10)
SEARCH_FROM = BASE_WINDOW[1] + 1        # -9; never search inside the baseline
MIN_CELL_N  = 200
SEP_LEVEL   = 0.15
SEP_RATE    = 0.05
HOLD        = 2

BURN_IN_YM = ["2024-01", "2024-02", "2024-03"]
NEW_ENTITY = ["cpty_new_out", "fin_new_out"]

# ── Peer groups ───────────────────────────────────────────────────────
PEER_DECILES  = 10          # size deciles of bal_live, recomputed each month
PEER_USE_SEG  = False       # True adds segment_desc; watch for thin cells
PEER_MIN_N    = 50

# ── Operating points ──────────────────────────────────────────────────
OP_THRESH   = [0.95, 0.90, 0.85, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10]
TARGET_PREC = 0.25
MIN_RECALL  = 0.20
NORM        = "peer"        # "peer" or "self" - which one drives the report

# ── THE TWELVE SIGNALS ────────────────────────────────────────────────
# rule: ratio = fell below a fraction of its reference
#       zero  = went to zero this month (the natural rule for a count)
#       sign  = crossed into negative territory
SIGNALS = [
 dict(n=1,  name="Stops taking on new trading partners",       feature="cpty_new_out",    rule="zero"),
 dict(n=2,  name="Stops paying anyone at an unfamiliar bank",   feature="fin_new_out",     rule="zero"),
 dict(n=3,  name="First rail to go — cheque",                   feature="amt_out_check",   rule="ratio"),
 dict(n=4,  name="Net flow turns against us",                   feature="net_flow",        rule="sign"),
 dict(n=5,  name="Their own customers stop paying them here",   feature="amt_in_internal", rule="ratio"),
 dict(n=6,  name="Spending through us falls",                   feature="amt_out",         rule="ratio"),
 dict(n=7,  name="Fewer payments, not just smaller",            feature="n_out",           rule="ratio",
                                                                companion="avg_ticket_out"),
 dict(n=8,  name="Inbound activity thins",                      feature="n_in",            rule="ratio"),
 dict(n=9,  name="What today's monitoring sees",                feature="bal_live",        rule="ratio"),
 dict(n=10, name="Payments to other PNC customers fall",        feature="amt_out_internal",rule="ratio"),
 dict(n=11, name="The relationship list itself shrinks",        feature="cpty_out_n",      rule="ratio"),
 dict(n=12, name="Banks they pay drop away",                    feature="fin_out_n",       rule="ratio"),
]
SIGNAL_FEATS = [s["feature"] for s in SIGNALS] + ["avg_ticket_out", "avg_ticket_in"]

RAILS = ["ach", "wire", "check", "card", "rtp", "other"]
RAIL_FEATS = [f"amt_out_{r}" for r in RAILS]

# Combined score: a customer-month scores 1 per signal firing at its
# chosen threshold. Deliberately an unweighted count, not a model - it is
# auditable line by line, which is what a first production rule must be.
SCORE_THRESH = 0.70
HTML_NAME    = "PKG_Attrition_Signals.html"

In [ ]:
# =====================================================================
# 1 · IMPORTS, HELPERS, SVG CHART PRIMITIVES
# =====================================================================
import warnings, html as _html, datetime as dt
import numpy as np, pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark import StorageLevel
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)

spark = (SparkSession.builder.appName("pkg_attrition_eda_v4")
         .config("spark.sql.shuffle.partitions", "400")
         .config("spark.sql.execution.arrow.pyspark.enabled", "false")
         .enableHiveSupport().getOrCreate())

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def hp(n):  return f"{HDFS_DIR.rstrip('/')}/{n}"
def v2(n):  return f"{HDFS_V2.rstrip('/')}/{n}"
def v3(n):  return f"{HDFS_V3.rstrip('/')}/{n}"

def _dec(sdf):
    o = sdf
    for c, t in sdf.dtypes:
        if t.startswith("decimal"): o = o.withColumn(c, F.col(c).cast("double"))
    return o

def disp(obj, title=None, n=None, save=None, transpose=False):
    n = MAX_ROWS if n is None else n
    out = _dec(obj).limit(n).toPandas() if hasattr(obj, "toPandas") else (
        obj.copy() if isinstance(obj, pd.DataFrame) else pd.DataFrame(obj))
    if save: out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title:
        display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;margin:10px 0 2px;"
                     f"color:#111'>{title}<span style='font-weight:400;color:#888'> &middot; "
                     f"{len(out)} rows</span></div>"))
    display(out.T if transpose else out)
    return out

def kv(d, title=None, save=None):
    return disp(pd.DataFrame({"metric": list(d.keys()), "value": list(d.values())}),
                title=title, n=len(d), save=save)

def pct(a, b): return float(a) / float(b) if b else float("nan")
def sdiv(a, b): return F.when(F.col(b) > 0, F.col(a) / F.col(b))

# ── inline SVG line chart, for the HTML report ────────────────────────
# No CDN, no JS. A bank laptop opens this file offline and it renders.
def svg_lines(df, ycols, title="", ylab="", W=560, H=210, pad=44,
              colors=("#C1440E", "#4A6FA5"), yzero=False, vline=0):
    d = df.dropna(subset=list(ycols), how="all")
    if d.empty: return "<div style='color:#999'>insufficient data</div>"
    xs = d.index.astype(float).values
    vals = np.concatenate([d[c].astype(float).dropna().values for c in ycols]) or np.array([0.0])
    lo, hi = float(np.nanmin(vals)), float(np.nanmax(vals))
    if yzero: lo = min(lo, 0.0)
    if hi == lo: hi = lo + 1e-9
    pad_y = (hi - lo) * 0.08; lo -= pad_y; hi += pad_y
    def X(v): return pad + (v - xs.min()) / max(xs.max() - xs.min(), 1e-9) * (W - pad - 14)
    def Y(v): return H - pad + 6 - (v - lo) / (hi - lo) * (H - pad - 22)
    p = [f"<svg viewBox='0 0 {W} {H}' width='100%' style='max-width:{W}px;font:11px IBM Plex Sans,sans-serif'>"]
    for gy in np.linspace(lo, hi, 5):
        p.append(f"<line x1='{pad}' y1='{Y(gy):.1f}' x2='{W-14}' y2='{Y(gy):.1f}' stroke='#EEE'/>")
        p.append(f"<text x='{pad-6}' y='{Y(gy)+3:.1f}' text-anchor='end' fill='#999'>{gy:,.2f}</text>")
    if vline is not None and xs.min() <= vline <= xs.max():
        p.append(f"<line x1='{X(vline):.1f}' y1='16' x2='{X(vline):.1f}' y2='{H-pad+6}' "
                 f"stroke='#BBB' stroke-dasharray='3,3'/>")
    for i, c in enumerate(ycols):
        s = d[c].astype(float)
        pts = " ".join(f"{X(x):.1f},{Y(v):.1f}" for x, v in zip(s.index.astype(float), s.values)
                       if pd.notna(v))
        if pts:
            p.append(f"<polyline points='{pts}' fill='none' stroke='{colors[i%len(colors)]}' stroke-width='2'/>")
            p.append(f"<circle cx='{W-150+i*78}' cy='12' r='4' fill='{colors[i%len(colors)]}'/>"
                     f"<text x='{W-142+i*78}' y='15' fill='#555'>{_html.escape(str(c))}</text>")
    for x in xs[::max(1, len(xs)//8)]:
        p.append(f"<text x='{X(x):.1f}' y='{H-pad+22}' text-anchor='middle' fill='#999'>{int(x)}</text>")
    p.append(f"<text x='{pad}' y='12' fill='#333' font-weight='600'>{_html.escape(title)}</text>")
    p.append(f"<text x='{W/2:.0f}' y='{H-4}' text-anchor='middle' fill='#AAA'>months before event</text>")
    p.append("</svg>")
    return "".join(p)

def svg_bars(labels, values, title="", W=560, H=210, pad=44, color="#C1440E"):
    if not len(values): return "<div style='color:#999'>insufficient data</div>"
    hi = max(max(values), 1e-9); n = len(values)
    bw = (W - pad - 14) / max(n, 1) * 0.68
    p = [f"<svg viewBox='0 0 {W} {H}' width='100%' style='max-width:{W}px;font:11px IBM Plex Sans,sans-serif'>"]
    p.append(f"<text x='{pad}' y='12' fill='#333' font-weight='600'>{_html.escape(title)}</text>")
    for i, (l, v) in enumerate(zip(labels, values)):
        x = pad + (W - pad - 14) * (i + 0.16) / n
        h = 0 if pd.isna(v) else (v / hi) * (H - pad - 26)
        p.append(f"<rect x='{x:.1f}' y='{H-pad+6-h:.1f}' width='{bw:.1f}' height='{max(h,0):.1f}' fill='{color}'/>")
        p.append(f"<text x='{x+bw/2:.1f}' y='{H-pad+20}' text-anchor='middle' fill='#999'>{_html.escape(str(l))}</text>")
        if pd.notna(v):
            p.append(f"<text x='{x+bw/2:.1f}' y='{H-pad+2-h:.1f}' text-anchor='middle' fill='#555'>{v:,.2f}</text>")
    p.append("</svg>")
    return "".join(p)

def tbl(df, cls="t"):
    return df.to_html(index=False, classes=cls, border=0, float_format=lambda v: f"{v:,.3f}",
                      na_rep="&mdash;", escape=False)

_F = OUT_DIR / "FINDINGS_v4.csv"
FINDINGS = pd.read_csv(_F).to_dict("records") if _F.exists() else []
def note(qid, question, answer, detail=""):
    global FINDINGS
    FINDINGS = [f for f in FINDINGS if f["id"] != qid]
    FINDINGS.append(dict(id=qid, question=question, answer=str(answer), detail=str(detail)))
    pd.DataFrame(FINDINGS).to_csv(_F, index=False)
print("helpers ready")

## 1 · Panel, derived signals, peer normalisation

In [ ]:
# =====================================================================
# 2 · PANEL + DERIVED SIGNALS + PEER NORMALISATION      [OUTPUT BLOCK 1]
# =====================================================================
acct_month = spark.read.parquet(v2("panel_account_month"))
cust_month = spark.read.parquet(v2("panel_customer_month"))
feat       = spark.read.parquet(v2("panel_pay_features"))
lab        = spark.read.parquet(v3("labels_customer")).persist(StorageLevel.DISK_ONLY)

# hard floor — nothing before 2024 is trusted, and nothing can be added
cust_month = cust_month.filter(F.col("ym") >= F.lit(DATE_START[:7]))
feat       = feat.filter(F.col("ym") >= F.lit(DATE_START[:7]))
for c in NEW_ENTITY:
    feat = feat.withColumn(c, F.when(F.col("ym").isin(*BURN_IN_YM), None).otherwise(F.col(c)))

panel = (cust_month.select("cust_pwr_id", "ym", "m_idx", "bal_live", "n_accts",
                           "segment_desc", "naics", "state")
         .join(feat.drop("ym"), ["cust_pwr_id", "m_idx"], "left")
         # SIGNAL 7 — fewer payments vs smaller payments needs the ticket
         .withColumn("avg_ticket_out", sdiv("amt_out", "n_out"))
         .withColumn("avg_ticket_in",  sdiv("amt_in",  "n_in")))

FEATS = sorted(set(SIGNAL_FEATS + RAIL_FEATS + ["amt_in", "cpty_out_n", "fin_out_n"])
               & set(panel.columns) | {"bal_live"})
_stack = ", ".join([f"'{f}', CAST({f} AS DOUBLE)" for f in FEATS])

# ── peer groups: size decile within the calendar month ────────────────
# Recomputed every month, so a customer that shrinks moves down the
# deciles with its peers rather than being scored against its own past.
wq = Window.partitionBy("ym")
pn = (panel.withColumn("bal_decile", F.ntile(PEER_DECILES).over(wq.orderBy(F.col("bal_live").asc_nulls_first())))
           .withColumn("peer_key", F.concat_ws("|", F.col("bal_decile").cast("string"),
                                               F.lit("SEG") if not PEER_USE_SEG else
                                               F.coalesce("segment_desc", F.lit("NA")))))
long_all = pn.select("cust_pwr_id", "m_idx", "ym", "peer_key",
                     F.expr(f"stack({len(FEATS)}, {_stack}) as (feature, value)"))

peer_med = (long_all.groupBy("ym", "peer_key", "feature")
            .agg(F.count("value").alias("peer_n"),
                 F.expr("percentile_approx(value, 0.5)").alias("peer_med"))
            .filter(F.col("peer_n") >= PEER_MIN_N))

long_all = (long_all.join(peer_med, ["ym", "peer_key", "feature"], "left")
            .withColumn("peer_idx", F.when(F.abs(F.col("peer_med")) > 1e-9,
                                           F.col("value") / F.col("peer_med")))
            ).persist(StorageLevel.DISK_ONLY)

disp(peer_med.groupBy("feature").agg(F.count("*").alias("peer_cells"),
                                     F.avg("peer_n").alias("mean_cell_n"),
                                     F.min("peer_n").alias("min_cell_n")).orderBy("feature"),
     title="1a &middot; Peer cells per feature (thin cells are dropped, not read)",
     n=40, save="v4_peer_cells")

kv({"customer-months": panel.count(),
    "features": len(FEATS),
    "peer grouping": f"{PEER_DECILES} balance deciles" + (" x segment" if PEER_USE_SEG else ""),
    "months": panel.select("ym").distinct().count(),
    "earliest month": panel.agg(F.min("ym")).collect()[0][0]},
   title="1b &middot; Panel", save="v4_panel")

note("PEER", "How is the permanent 2024 floor handled?",
     "Peer-relative normalisation: size decile within the same calendar month",
     "A 12-month self-baseline costs ~4,200 of 16,384 attriters and can never be recovered, "
     "because pre-2024 data is not trusted. Peer-relative needs no pre-window, uses every "
     "attriter, and cancels the new-entity burn-in because the peer median is inflated too.")

## 2 · Event study under both normalisations

In [ ]:
# =====================================================================
# 3 · EVENT STUDY UNDER BOTH NORMALISATIONS             [OUTPUT BLOCK 2]
# =====================================================================
# Agreement between self-baseline and peer-relative is the sensitivity
# check the short panel makes necessary. Where they disagree, peer wins:
# it uses every attriter and needs no pre-window.

def cohorts(defn, need_pre):
    ev = f"q_{defn}"
    a = (lab.filter(F.col(ev).isNotNull())
         .select("cust_pwr_id", F.col(ev).alias("event_m"),
                 F.col("first_live_m").alias("first_m"), "last_m")
         .withColumn("cohort", F.lit("attriter")))
    dr = [r.event_m for r in a.select("event_m").limit(3000).collect()]
    dr = dr[::max(1, len(dr)//300)][:300] or [0]
    arr = F.array(*[F.lit(int(x)) for x in dr])
    c = (lab.filter(F.col("q_A_full_exit").isNull() & F.col("q_B_bal_exit").isNull())
         .withColumn("event_m", F.element_at(arr, (F.abs(F.hash("cust_pwr_id")) % len(dr)) + 1))
         .withColumn("first_m", F.coalesce("first_live_m", "first_m"))
         .select("cust_pwr_id", "event_m", "first_m", "last_m")
         .withColumn("cohort", F.lit("stayer")))
    out = a.unionByName(c).filter(F.col("last_m") >= F.col("event_m"))
    if need_pre:                      # self-baseline needs a clean pre-window
        out = out.filter(F.col("first_m") <= F.col("event_m") - EVENT_PRE)
    return out

def build(defn):
    res = {}
    for mode, need_pre in [("peer", False), ("self", True)]:
        co = cohorts(defn, need_pre)
        es = (long_all.join(co, "cust_pwr_id", "inner")
              .withColumn("rel_m", F.col("m_idx") - F.col("event_m"))
              .filter(F.col("rel_m").between(-EVENT_PRE, EVENT_POST)))
        if mode == "self":
            base = (es.filter(F.col("rel_m").between(*BASE_WINDOW))
                    .groupBy("cust_pwr_id", "feature").agg(F.avg("value").alias("base")))
            es = (es.join(base, ["cust_pwr_id", "feature"], "left")
                  .withColumn("idx", F.when(F.abs(F.col("base")) > 1e-9,
                                            F.col("value") / F.col("base"))))
        else:
            es = es.withColumn("idx", F.col("peer_idx"))
        es = es.persist(StorageLevel.DISK_ONLY)
        cur = (es.groupBy("feature", "cohort", "rel_m").agg(
                   F.count("*").alias("n"),
                   F.expr("percentile_approx(idx, 0.5)").alias("med_idx"),
                   F.avg(F.when(F.col("value").isNotNull(),
                                (F.col("value") > 0).cast("double"))).alias("rate_any"),
                   F.avg(F.when(F.col("value").isNotNull(),
                                (F.col("value") < 0).cast("double"))).alias("rate_neg"),
                   F.avg("value").alias("mean_value"),
                   F.expr("percentile_approx(value, 0.5)").alias("med_value"))).toPandas()
        cur.loc[cur.n < MIN_CELL_N, ["med_idx", "rate_any", "rate_neg", "mean_value", "med_value"]] = np.nan
        cur.to_csv(OUT_DIR / f"v4_curves_{defn}_{mode}.csv", index=False)
        res[mode] = dict(es=es, cur=cur,
                         n_att=co.filter("cohort='attriter'").count(),
                         n_sta=co.filter("cohort='stayer'").count())
    return res

B = {d: build(d) for d in STUDY_DEFS}
disp(pd.DataFrame([dict(definition=d, normalisation=m, attriters=B[d][m]["n_att"],
                        stayers=B[d][m]["n_sta"]) for d in STUDY_DEFS for m in ("peer", "self")]),
     title="2a &middot; Cohort sizes — peer-relative recovers the attriters the self-baseline drops",
     save="v4_cohorts")

# ── separation, both normalisations ───────────────────────────────────
def seps(cur):
    rows = []
    for f, g in cur.groupby("feature"):
        col, thr, kind = ("rate_any", SEP_RATE, "rate") if f in NEW_ENTITY else ("med_idx", SEP_LEVEL, "level")
        w = (g.pivot_table(index="rel_m", columns="cohort", values=col)
               .reindex(columns=["attriter", "stayer"]).dropna().sort_index())
        if w.empty: continue
        gap = (w.attriter - w.stayer).abs(); s = gap[gap.index >= SEARCH_FROM]
        sep, run = None, 0
        for rm, v in s.items():
            run = run + 1 if v > thr else 0
            if run >= HOLD: sep = rm - HOLD + 1; break
        rows.append(dict(feature=f, kind=kind, sep_rel_m=sep,
                         lead=None if sep is None else -sep, max_gap=round(s.max(), 3) if len(s) else np.nan))
    return pd.DataFrame(rows)

cmpt = (seps(B["A_full_exit"]["peer"]["cur"]).rename(columns={"sep_rel_m": "peer_rel_m", "lead": "peer_lead"})
        .merge(seps(B["A_full_exit"]["self"]["cur"])[["feature", "sep_rel_m", "lead"]]
               .rename(columns={"sep_rel_m": "self_rel_m", "lead": "self_lead"}), on="feature", how="outer")
        .assign(agree=lambda d: (d.peer_lead - d.self_lead).abs() <= 2)
        .sort_values("peer_rel_m", na_position="last"))
disp(cmpt, title="2b &middot; Lead under both normalisations — agreement is the sensitivity check",
     n=40, save="v4_lead_compare")
note("SENS", "Do the two normalisations agree on the ordering?",
     f"{int(cmpt.agree.sum())} of {len(cmpt)} features agree within 2 months",
     "Peer-relative is the reported one: no pre-window, every attriter, burn-in cancelled.")

## 3 · Operating points for all twelve

In [ ]:
# =====================================================================
# 4 · OPERATING POINTS FOR ALL TWELVE                   [OUTPUT BLOCK 3]
# =====================================================================
# FIXES THE v3 GAP. v3 measured precision on eight dense features and left
# out cpty_new_out and fin_new_out - the two that separate EARLIEST. A
# count feature needs its own rule: "went to zero this month" is the
# natural operating rule for "stopped taking on new partners", and a ratio
# threshold cannot express it.

EVAL_M = cust_month.agg((F.max("m_idx") - F.min("m_idx") + 1)).collect()[0][0] - 12
N_EV   = lab.count()
PREV   = {d: pct(lab.filter(F.col(f"q_{d}").isNotNull()).count(), N_EV * EVAL_M) for d in STUDY_DEFS}
kv({f"monthly hazard, {d}": PREV[d] for d in STUDY_DEFS} | {"evaluable months": EVAL_M},
   title="3a &middot; Base rate an alert queue actually faces", save="v4_prev")

RULE = {s["feature"]: s["rule"] for s in SIGNALS}

def op(defn, mode=NORM):
    es, p = B[defn][mode]["es"], PREV[defn]
    ratio_f = [f for f, r in RULE.items() if r == "ratio"] + ["avg_ticket_out"]
    aggs = [F.count("*").alias("n"), F.sum(F.col("idx").isNotNull().cast("int")).alias("n_idx"),
            F.sum(F.col("value").isNotNull().cast("int")).alias("n_val"),
            F.sum(((F.col("value").isNotNull()) & (F.col("value") <= 0)).cast("int")).alias("hit_zero"),
            F.sum(((F.col("value").isNotNull()) & (F.col("value") < 0)).cast("int")).alias("hit_sign")]
    aggs += [F.sum((F.col("idx") < t).cast("int")).alias(f"lt{i}") for i, t in enumerate(OP_THRESH)]
    raw = (es.filter(F.col("feature").isin(*RULE.keys()) & F.col("rel_m").between(-EVENT_PRE, -1))
             .groupBy("feature", "rel_m", "cohort").agg(*aggs)).toPandas()
    out = []
    for (f, rm), g in raw.groupby(["feature", "rel_m"]):
        a, s = g[g.cohort == "attriter"], g[g.cohort == "stayer"]
        if a.empty or s.empty: continue
        a, s = a.iloc[0], s.iloc[0]
        rule = RULE.get(f, "ratio")
        cands = ([("zero", "hit_zero", a.n_val, s.n_val)] if rule == "zero" else
                 [("negative", "hit_sign", a.n_val, s.n_val)] if rule == "sign" else
                 [(t, f"lt{i}", a.n_idx, s.n_idx) for i, t in enumerate(OP_THRESH)])
        for thr, col, na, ns in cands:
            if na < MIN_CELL_N or ns < MIN_CELL_N: continue
            rec, fpr = a[col] / na, s[col] / ns
            den = p * rec + (1 - p) * fpr
            pr = (p * rec / den) if den > 0 else np.nan
            out.append(dict(feature=f, rel_m=int(rm), rule=rule, threshold=thr,
                            recall=rec, fpr=fpr, precision=pr,
                            lift=(pr / p) if pr == pr else np.nan,
                            alerts_per_tp=(1 / pr) if pr and pr > 0 else np.nan,
                            n_attriter=int(na), n_stayer=int(ns)))
    return pd.DataFrame(out)

OP = {d: op(d) for d in STUDY_DEFS}
for d in STUDY_DEFS: OP[d].to_csv(OUT_DIR / f"v4_operating_{d}.csv", index=False)

def best_per(dfp, target=TARGET_PREC, min_rec=MIN_RECALL):
    rows = []
    for f, g in dfp.groupby("feature"):
        ok = g[(g.precision >= target) & (g.recall >= min_rec)]
        top = g.loc[g.precision.idxmax()] if not g.empty else None
        e = ok.loc[ok.rel_m.idxmin()] if not ok.empty else None
        rows.append(dict(feature=f,
                         earliest_usable_rel_m=None if e is None else int(e.rel_m),
                         usable_threshold=None if e is None else e.threshold,
                         usable_recall=None if e is None else round(e.recall, 3),
                         usable_precision=None if e is None else round(e.precision, 3),
                         best_precision=None if top is None else round(top.precision, 3),
                         best_lift=None if top is None else round(top.lift, 1),
                         best_at_rel_m=None if top is None else int(top.rel_m),
                         best_recall=None if top is None else round(top.recall, 3)))
    return pd.DataFrame(rows).sort_values("best_lift", ascending=False, na_position="last")

BEST = {d: best_per(OP[d]) for d in STUDY_DEFS}
disp(BEST["A_full_exit"], title=f"3b &middot; Every signal scored — including the two v3 never "
                                f"measured (target precision {TARGET_PREC:.0%})",
     n=25, save="v4_best_A")
disp(BEST["B_bal_exit"], title="3c &middot; Same, for the shell-account population",
     n=25, save="v4_best_B")

_u = BEST["A_full_exit"].dropna(subset=["earliest_usable_rel_m"])
note("OPALL", "With every signal measured, does any single rule work?",
     ("NO - none reaches the target" if _u.empty else
      f"{_u.iloc[0].feature} at rel_m {int(_u.iloc[0].earliest_usable_rel_m)}"),
     f"Best lift over the {PREV['A_full_exit']:.3%} base rate: "
     f"{BEST['A_full_exit'].best_lift.max():.0f}x. Lift, not precision, is the honest read on "
     "whether a signal carries information.")

## 4 · Deep dives: rails, ticket size, net-flow sign, the incumbent

In [ ]:
# =====================================================================
# 5 · DEEP DIVES THE TWELVE-SIGNAL LIST NEEDS           [OUTPUT BLOCK 4]
# =====================================================================
CUR = B["A_full_exit"][NORM]["cur"]
ES  = B["A_full_exit"][NORM]["es"]

def pv(cur, feats, col="med_idx"):
    return (cur[cur.feature.isin(feats)]
            .pivot_table(index="rel_m", columns=["feature", "cohort"], values=col))

# ── SIGNAL 3 · which rail goes first ──────────────────────────────────
rail_rows = []
for f in [r for r in RAIL_FEATS if r in set(CUR.feature)]:
    g = CUR[CUR.feature == f].pivot_table(index="rel_m", columns="cohort", values="med_idx")
    if not {"attriter", "stayer"} <= set(g.columns): continue
    gap = (g.attriter - g.stayer).abs(); s = gap[gap.index >= SEARCH_FROM]
    sep, run = None, 0
    for rm, v in s.items():
        run = run + 1 if v > SEP_LEVEL else 0
        if run >= HOLD: sep = rm - HOLD + 1; break
    rail_rows.append(dict(rail=f.replace("amt_out_", "").upper(), feature=f, sep_rel_m=sep,
                          attr_at_minus6=round(g.attriter.get(-6, np.nan), 3),
                          attr_at_minus3=round(g.attriter.get(-3, np.nan), 3),
                          max_gap=round(s.max(), 3) if len(s) else np.nan))
RAIL_ORDER = pd.DataFrame(rail_rows).sort_values("sep_rel_m", na_position="last")
disp(RAIL_ORDER, title="4a &middot; SIGNAL 3 — which rail actually goes first", save="v4_rail_order")

# ── SIGNAL 7 · fewer payments, or smaller ones ────────────────────────
TICKET = pv(CUR, ["n_out", "avg_ticket_out", "amt_out"]).round(3)
disp(TICKET.reset_index(), title="4b &middot; SIGNAL 7 — count vs ticket size. If n_out falls "
                                 "faster than avg_ticket_out, they make FEWER payments, not smaller ones",
     n=EVENT_PRE + EVENT_POST + 1, save="v4_ticket")
_t = TICKET.dropna()
DECOMP = None
if not _t.empty and ("n_out", "attriter") in _t.columns:
    DECOMP = pd.DataFrame({
        "rel_m": _t.index,
        "count_ratio":  _t[("n_out", "attriter")].values,
        "ticket_ratio": _t[("avg_ticket_out", "attriter")].values,
        "amount_ratio": _t[("amt_out", "attriter")].values})
    DECOMP["driver"] = np.where(DECOMP.count_ratio < DECOMP.ticket_ratio, "FEWER payments", "SMALLER payments")
    disp(DECOMP, title="4c &middot; Which one drives the fall, month by month",
         n=EVENT_PRE + EVENT_POST + 1, save="v4_ticket_decomp")

# ── SIGNAL 4 · does net flow actually turn negative ───────────────────
NETSIGN = (CUR[CUR.feature == "net_flow"]
           .pivot_table(index="rel_m", columns="cohort", values="rate_neg").round(3))
disp(NETSIGN.reset_index(), title="4d &middot; SIGNAL 4 — share of customers whose net flow is "
                                  "NEGATIVE (more out than in)",
     n=EVENT_PRE + EVENT_POST + 1, save="v4_netflow_sign")

# ── SIGNAL 9 · what today's monitoring sees, as a benchmark ───────────
# The 30% balance rule is the incumbent. Everything else is only
# interesting to the extent it fires earlier than this.
bench = (lab.filter(F.col("q_A_full_exit").isNotNull())
         .withColumn("p30_lead", F.col("q_A_full_exit") - F.col("m_C_p30"))
         .select("cust_pwr_id", "p30_lead", "m_C_p30", "q_A_full_exit"))
bp = bench.toPandas()
BENCH = pd.DataFrame({
    "metric": ["attriters", "…the 30% rule ever fires for", "…it never fires for",
               "median months of warning it gives", "p25", "p75"],
    "value": [len(bp), int(bp.p30_lead.notna().sum()), int(bp.p30_lead.isna().sum()),
              bp.p30_lead.median(), bp.p30_lead.quantile(.25), bp.p30_lead.quantile(.75)]})
disp(BENCH, title="4e &middot; SIGNAL 9 — the incumbent 30% rule, as a benchmark", save="v4_benchmark")

_med_p30 = bp.p30_lead.median()
note("BENCH", "How much warning does today's monitoring give?",
     f"median {_med_p30:.0f} months, and it never fires for {int(bp.p30_lead.isna().sum()):,} attriters",
     "Every other signal is only worth deploying to the extent it beats this. Note the 30% rule "
     "also fired on 43,143 customers who never left (v2 5a), so its warning is cheap.")

## 5 · Do the signals stack?

In [ ]:
# =====================================================================
# 6 · DOES COMBINING THE SIGNALS HELP?                  [OUTPUT BLOCK 5]
# =====================================================================
# v3 found no single feature clears the precision bar. The natural next
# question is whether they stack. This is an UNWEIGHTED COUNT of signals
# firing, not a fitted model - auditable line by line, which is what a
# first production rule has to be. A fitted hazard model is the step
# after this, and this is its baseline.

flag = (F.when(F.col("feature").isin(*NEW_ENTITY) & F.col("value").isNotNull(),
               (F.col("value") <= 0).cast("int"))
         .when((F.col("feature") == "net_flow") & F.col("value").isNotNull(),
               (F.col("value") < 0).cast("int"))
         .when(F.col("idx").isNotNull(), (F.col("idx") < SCORE_THRESH).cast("int"))
         .otherwise(None))

sc = (ES.filter(F.col("feature").isin(*RULE.keys()))
      .withColumn("fired", flag)
      .groupBy("cust_pwr_id", "cohort", "rel_m")
      .agg(F.sum("fired").alias("n_fired"), F.count("fired").alias("n_measured"))
      .filter(F.col("n_measured") >= 6)).persist(StorageLevel.DISK_ONLY)

p = PREV["A_full_exit"]
raw = (sc.filter(F.col("rel_m").between(-EVENT_PRE, -1))
       .groupBy("rel_m", "cohort")
       .agg(F.count("*").alias("n"),
            *[F.sum((F.col("n_fired") >= k).cast("int")).alias(f"ge{k}") for k in range(1, 10)])
       ).toPandas()

rows = []
for rm, g in raw.groupby("rel_m"):
    a, s = g[g.cohort == "attriter"], g[g.cohort == "stayer"]
    if a.empty or s.empty: continue
    a, s = a.iloc[0], s.iloc[0]
    if a.n < MIN_CELL_N or s.n < MIN_CELL_N: continue
    for k in range(1, 10):
        rec, fpr = a[f"ge{k}"] / a.n, s[f"ge{k}"] / s.n
        den = p * rec + (1 - p) * fpr
        pr = (p * rec / den) if den > 0 else np.nan
        rows.append(dict(rel_m=int(rm), min_signals=k, recall=rec, fpr=fpr, precision=pr,
                         lift=pr / p if pr == pr else np.nan,
                         alerts_per_tp=1 / pr if pr and pr > 0 else np.nan))
SCORE = pd.DataFrame(rows)
SCORE.to_csv(OUT_DIR / "v4_score.csv", index=False)

disp(SCORE.pivot_table(index="min_signals", columns="rel_m", values="precision").round(3).reset_index(),
     title=f"5a &middot; Precision by how many signals fire (threshold {SCORE_THRESH})",
     n=12, save="v4_score_precision")
disp(SCORE.pivot_table(index="min_signals", columns="rel_m", values="recall").round(3).reset_index(),
     title="5b &middot; Recall — the trade against 5a", n=12, save="v4_score_recall")

_ok = SCORE[(SCORE.precision >= TARGET_PREC) & (SCORE.recall >= MIN_RECALL)]
BESTSCORE = None
if not _ok.empty:
    BESTSCORE = _ok.loc[_ok.rel_m.idxmin()]
_single = BEST["A_full_exit"].best_precision.max()
kv({"best single-signal precision": _single,
    "best combined precision": round(SCORE.precision.max(), 3) if len(SCORE) else None,
    "combining helps?": ("yes" if len(SCORE) and SCORE.precision.max() > _single else "no"),
    "earliest usable combined rule":
        None if BESTSCORE is None else
        f"{int(BESTSCORE.min_signals)}+ signals at rel_m {int(BESTSCORE.rel_m)} "
        f"(recall {BESTSCORE.recall:.0%}, precision {BESTSCORE.precision:.0%})"},
   title="5c &middot; Does stacking beat the best single rule?", save="v4_score_verdict")

note("SCORE", "Does combining the twelve signals beat any one of them?",
     ("no usable combined rule" if BESTSCORE is None else
      f"{int(BESTSCORE.min_signals)}+ signals at rel_m {int(BESTSCORE.rel_m)}: "
      f"recall {BESTSCORE.recall:.0%}, precision {BESTSCORE.precision:.0%}"),
     "Unweighted count, no fitting. If this beats the best single rule the case for a fitted "
     "hazard model is made; if it does not, the signals are collinear and a model will not "
     "rescue them either.")

## 6 · Generate the HTML report

In [ ]:
# =====================================================================
# 7 · THE TWELVE-SIGNAL HTML REPORT
# =====================================================================
# Self-contained: inline SVG, no CDN, no JavaScript. Opens offline.

CSS = """
:root{--ink:#16181D;--mut:#6B7280;--line:#E5E7EB;--acc:#C1440E;--acc2:#4A6FA5;--bg:#FCFCFD}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--ink);
 font:15px/1.65 "IBM Plex Sans",-apple-system,Segoe UI,sans-serif}
.wrap{max-width:1080px;margin:0 auto;padding:48px 32px 96px}
h1{font-size:30px;font-weight:600;margin:0 0 6px;letter-spacing:-.02em}
h2{font-size:20px;font-weight:600;margin:0 0 4px;letter-spacing:-.01em}
.sub{color:var(--mut);font-size:14px;margin:0 0 36px}
.sig{border:1px solid var(--line);border-radius:10px;background:#fff;padding:24px 26px;margin:0 0 20px}
.sig-h{display:flex;align-items:baseline;gap:12px;border-bottom:1px solid var(--line);
 padding-bottom:12px;margin-bottom:16px}
.num{font:600 13px "IBM Plex Mono",monospace;color:#fff;background:var(--acc);
 border-radius:4px;padding:3px 8px;flex:none}
.feat{font:12px "IBM Plex Mono",monospace;color:var(--mut);margin-left:auto}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px;margin:14px 0}
.card{border:1px solid var(--line);border-radius:8px;padding:11px 13px;background:var(--bg)}
.card .k{font-size:11px;color:var(--mut);text-transform:uppercase;letter-spacing:.05em}
.card .v{font:600 21px "IBM Plex Mono",monospace;margin-top:3px}
.card .v.hi{color:var(--acc)}
table.t{border-collapse:collapse;width:100%;font-size:13px;margin:12px 0}
table.t th{text-align:left;font-weight:600;color:var(--mut);font-size:11px;
 text-transform:uppercase;letter-spacing:.04em;border-bottom:1px solid var(--line);padding:7px 10px}
table.t td{padding:6px 10px;border-bottom:1px solid #F3F4F6;font-variant-numeric:tabular-nums}
table.t tr:hover td{background:#FAFAFB}
.read{background:#FFF8F4;border-left:3px solid var(--acc);padding:11px 15px;margin:14px 0;font-size:14px}
.warn{background:#F5F7FA;border-left:3px solid var(--acc2);padding:11px 15px;margin:14px 0;font-size:14px}
.charts{display:grid;grid-template-columns:repeat(auto-fit,minmax(330px,1fr));gap:18px;margin:16px 0}
footer{color:var(--mut);font-size:12px;border-top:1px solid var(--line);padding-top:18px;margin-top:40px}
"""

def _cards(items):
    return ("<div class='grid'>" + "".join(
        f"<div class='card'><div class='k'>{_html.escape(k)}</div>"
        f"<div class='v{' hi' if hi else ''}'>{v}</div></div>" for k, v, hi in items) + "</div>")

def _fmt(v, s="{:.3f}"):
    return "&mdash;" if v is None or (isinstance(v, float) and pd.isna(v)) else s.format(v)

def signal_block(s):
    f, best = s["feature"], BEST["A_full_exit"]
    row = best[best.feature == f]
    row = row.iloc[0] if len(row) else None
    sep = cmpt[cmpt.feature == f]
    sep = sep.iloc[0] if len(sep) else None

    col = "rate_any" if f in NEW_ENTITY else ("rate_neg" if s["rule"] == "sign" else "med_idx")
    cur = CUR[CUR.feature == f].pivot_table(index="rel_m", columns="cohort", values=col)
    cur = cur.reindex(columns=[c for c in ("attriter", "stayer") if c in cur.columns])
    ylab = {"rate_any": "share with any", "rate_neg": "share negative"}.get(col, "vs peer median")
    chart = svg_lines(cur, list(cur.columns), title=ylab, yzero=(col != "med_idx"))

    opf = OP["A_full_exit"]; opf = opf[opf.feature == f]
    lift_by_m = (opf.groupby("rel_m").lift.max().sort_index()
                 if len(opf) else pd.Series(dtype=float))
    chart2 = svg_bars([str(int(i)) for i in lift_by_m.index], list(lift_by_m.values),
                      title="best lift over base rate, by month") if len(lift_by_m) else ""

    cards = [("separates at", f"{int(sep.peer_rel_m)} mo" if sep is not None and pd.notna(sep.peer_rel_m) else "&mdash;",
              True),
             ("best lift", _fmt(None if row is None else row.best_lift, "{:.0f}&times;"), True),
             ("best precision", _fmt(None if row is None else row.best_precision), False),
             ("recall there", _fmt(None if row is None else row.best_recall), False),
             ("operating rule", _html.escape(s["rule"]), False)]

    extra = ""
    if s["n"] == 3 and RAIL_ORDER is not None and len(RAIL_ORDER):
        extra += "<h3 style='font-size:14px;margin:18px 0 4px'>Rails ranked by when each one goes</h3>" + tbl(RAIL_ORDER)
    if s["n"] == 7 and DECOMP is not None:
        extra += ("<h3 style='font-size:14px;margin:18px 0 4px'>Fewer, or smaller?</h3>" +
                  tbl(DECOMP.round(3)) +
                  "<div class='read'>Where <b>count_ratio</b> sits below <b>ticket_ratio</b>, they are "
                  "making <b>fewer</b> payments of much the same size &mdash; the relationship is "
                  "being wound down, not the business shrinking.</div>")
    if s["n"] == 4 and len(NETSIGN):
        extra += "<h3 style='font-size:14px;margin:18px 0 4px'>Share with negative net flow</h3>" + tbl(NETSIGN.reset_index().round(3))
    if s["n"] == 9:
        extra += ("<h3 style='font-size:14px;margin:18px 0 4px'>The incumbent, measured</h3>" + tbl(BENCH) +
                  "<div class='warn'>Everything above is only worth deploying to the extent it beats "
                  "this &mdash; and the 30% rule also fired on 43,143 customers who never left.</div>")

    return (f"<section class='sig'><div class='sig-h'><span class='num'>{s['n']:02d}</span>"
            f"<h2>{_html.escape(s['name'])}</h2><span class='feat'>{_html.escape(f)}</span></div>"
            + _cards(cards)
            + f"<div class='charts'><div>{chart}</div><div>{chart2}</div></div>" + extra + "</section>")

_lead_tbl = cmpt[["feature", "peer_rel_m", "peer_lead", "self_rel_m", "self_lead", "agree"]].copy()
_score_tbl = (SCORE.pivot_table(index="min_signals", columns="rel_m", values="precision")
              .round(3).reset_index() if len(SCORE) else pd.DataFrame())

doc = f"""<!doctype html><html><head><meta charset="utf-8">
<title>PKG — Twelve early-warning signals</title><style>{CSS}</style></head><body><div class="wrap">
<h1>Twelve early-warning signals</h1>
<p class="sub">Payment Knowledge Graph &middot; PNC Treasury Management &middot; Data Science
&middot; generated {dt.date.today().isoformat()} &middot; cohort <b>A_full_exit</b>,
{B['A_full_exit'][NORM]['n_att']:,} attriters vs {B['A_full_exit'][NORM]['n_sta']:,} stayers
&middot; {NORM}-relative normalisation</p>

<div class="warn"><b>How to read this.</b> Every chart is months before the customer leaves; 0 is the
exit month. Lines are the cohort median against its size-decile peers in the same calendar month, so
1.0 means "normal for a customer this size". <b>Lift</b> is how much more likely a flagged customer is
to leave than a random one &mdash; the base rate is {PREV['A_full_exit']:.2%} a month, so a lift of
20&times; means roughly 1 in 5 rather than 1 in 110. Precision looks low because the base rate is low;
lift is the honest read.</div>

<div class="read"><b>Data floor.</b> Nothing before 2024 is trusted and none can be added, so the panel
is 31 months. That is why these figures are peer-relative rather than measured against each customer's
own past &mdash; a 12-month self-baseline would discard about a quarter of the attriters permanently.</div>

<h2 style="margin:34px 0 8px">Ordering: what moves first</h2>
{tbl(_lead_tbl)}
<div class="read">Agreement between the two normalisations is the sensitivity check the short panel
forces on us. Where they disagree, peer-relative is reported.</div>

{''.join(signal_block(s) for s in SIGNALS)}

<section class="sig"><div class="sig-h"><span class="num">&Sigma;</span>
<h2>Do they stack?</h2><span class="feat">unweighted count, threshold {SCORE_THRESH}</span></div>
{tbl(_score_tbl)}
<div class="read">Precision by how many of the twelve fire at once, across months before exit.
This is a plain count with no fitting &mdash; auditable line by line. If it beats the best single
rule, the case for a fitted hazard model is made. If not, the signals are collinear and a model will
not rescue them either.</div></section>

<footer>Generated by <code>pkg_attrition_eda_v4.ipynb</code>. Underlying tables in
<code>{OUT_DIR}</code>. Internal &mdash; PNC Treasury Management, Data Science.</footer>
</div></body></html>"""

(OUT_DIR / HTML_NAME).write_text(doc, encoding="utf-8")
print("wrote", OUT_DIR / HTML_NAME, f"({len(doc):,} bytes)")
display(HTML(f"<a href='{HTML_NAME}' target='_blank' style='font:600 14px IBM Plex Sans'>"
             f"open {HTML_NAME}</a>"))

reg = pd.DataFrame(FINDINGS)
disp(reg, title="6 &middot; v4 findings", n=40, save="FINDINGS_v4")
print("\nLocal:", OUT_DIR)
for f in sorted(OUT_DIR.glob("*")): print("  ", f.name)

---

## After this run

1. **Read §2b first.** If peer and self disagree by more than two months on any signal,
   that signal's position in the twelve is not settled and the HTML overstates it.
2. **§4 replaces v3's §5c.** Judge signals on **lift**, not precision — at a 0.9% monthly
   base rate, 20% precision is a 22× lift and a perfectly good queue.
3. **§6 is the go/no-go for modelling.** If stacking beats the best single rule, build the
   discrete-time hazard model (rolling-origin split on `m_idx`, nothing at or after t+1 in
   the feature set, 158,066 censored accounts make hazard the right frame). If it doesn't,
   the signals are collinear and a model won't rescue them — ship the best single rule and
   spend the effort on counterparty data instead.
4. **Signals 1 and 2 are the ones to watch.** They separate earliest and they are the
   *absence* of behaviour — no new partners, no new banks. Absence is cheap to monitor and
   hard to game, but it is also easy to confuse with a quiet month, which is exactly what
   the zero-rule precision in §4 measures.